# HAIR box-first annotation for Ultralytics Platform

Upload a rebuilt `hair_colab_runtime.zip` through the Colab Enterprise Files pane, then run each cell in order. Generic YOLOE localization proposes boxes; OpenCLIP ranks every enabled SKU. Low-score or ambiguous boxes become class 89, `Needs Review`. Every generated box requires human review. Download both result ZIPs before deleting the ephemeral runtime. The real-data runtime ZIP must first be built from all verified source inputs; it is not supplied by this checkout.

In [ ]:
from pathlib import Path
import shutil
import sys

archives = sorted(Path.cwd().rglob("hair_colab_runtime.zip"))
if len(archives) != 1:
    raise RuntimeError(f"Expected one uploaded hair_colab_runtime.zip, found {len(archives)}")
workspace = Path.cwd() / "hair_colab_workspace"
if workspace.exists():
    raise RuntimeError(f"Workspace already exists: {workspace}")
shutil.unpack_archive(archives[0], workspace)
project = workspace / "hair_colab"
sys.path.insert(0, str(project))
print(f"Project extracted to {project}")

In [ ]:
import colab_runtime
colab_runtime.environment_report()

In [ ]:
import subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], cwd=project, check=True)

In [ ]:
subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=project, check=True)

In [ ]:
subprocess.run([sys.executable, "yoloe_autolabel.py", "--config", "config.yaml", "--validate-only", "--require-enabled-count", "89"], cwd=project, check=True)

In [ ]:
pilot_config = colab_runtime.write_pilot_config(project, max_images=10)
subprocess.run([sys.executable, "yoloe_autolabel.py", "--config", pilot_config.name, "--require-enabled-count", "89"], cwd=project, check=True)

In [ ]:
import json
from IPython.display import display
from PIL import Image as PILImage
pilot_raw = project / "pilot_output" / "raw_predictions"
pilot_run = json.loads((pilot_raw / "run.json").read_text(encoding="utf-8"))
print(json.dumps(pilot_run, indent=2, ensure_ascii=False))
totals = pilot_run["totals"]
candidate_count = totals["candidate_count"]
pilot_diagnostics = {
    "geometry_rejections": totals["geometry_rejections"],
    "duplicates_removed": totals["duplicates_removed"],
    "zero_candidate_images": sum(record["candidate_count"] == 0 for record in pilot_run["images"].values()),
    "needs_review_ratio": totals["needs_review_count"] / candidate_count if candidate_count else None,
    "top_score_distributions": pilot_run["score_summaries"],
}
print(json.dumps(pilot_diagnostics, indent=2))
preview_paths = sorted((pilot_raw / "previews").glob("*"))[:10]
if preview_paths:
    columns, thumb_width, thumb_height = 2, 800, 500
    grid = PILImage.new("RGB", (columns * thumb_width, ((len(preview_paths) + columns - 1) // columns) * thumb_height), "white")
    for index, preview_path in enumerate(preview_paths):
        with PILImage.open(preview_path) as preview:
            thumbnail = preview.convert("RGB")
            thumbnail.thumbnail((thumb_width, thumb_height))
            grid.paste(thumbnail, ((index % columns) * thumb_width, (index // columns) * thumb_height))
    display(grid)
else:
    print("No previews generated; inspect run errors and zero-candidate images before continuing.")

## Full run
The pilot uses all 89 enabled SKU identities but only ten shelf images. Inspect original images alongside previews for box tightness, misses, wrong SKU suggestions, and the `Needs Review` rate; score summaries are ranking diagnostics, not calibrated probabilities. Check geometry rejections, duplicates, zero-candidate images, and any run errors. Adjust references or thresholds and repeat the pilot if necessary. Run the next cell only after this review is acceptable. It processes all 632 shelf images against all 89 enabled classes.

In [ ]:
subprocess.run([sys.executable, "yoloe_autolabel.py", "--config", "config.yaml", "--require-enabled-count", "89"], cwd=project, check=True)

In [ ]:
platform_zip, review_zip = colab_runtime.package_results(
    project,
    workspace.parent / "hair_ultralytics_platform.zip",
    workspace.parent / "hair_annotation_review.zip",
)
print(platform_zip)
print(review_zip)
print("Download both ZIPs from the Files pane before deleting the runtime. Existing ZIPs are protected unless overwrite=True is explicitly requested.")

## Download and review
`hair_ultralytics_platform.zip` goes to Ultralytics Platform. It contains original image bytes, numeric YOLO labels, and `data.yaml` mapping permanent IDs 0–88 to English SKU names plus temporary class 89, `Needs Review`. `hair_annotation_review.zip` stays beside the reviewer: use its ranked suggestions, barcode/Thai-name manifest, metadata, previews, and diagnostics. Neither ZIP is approved training truth.

Review every box: (1) reassign `Needs Review` boxes to the right SKU or remove false positives; (2) fix box geometry; (3) verify accepted SKU suggestions; (4) add missed products; (5) ensure class 89 has no remaining boxes; (6) only then delete class 89 before training the permanent 89-class detector. Never renumber permanent classes.

First inference may download `yoloe-26l-seg.pt`, the YOLOE26 text encoder `mobileclip2_b.ts`, and OpenCLIP `ViT-B-32` / `laion2b_s34b_b79k` weights. YOLOE class setup may also install the `ultralytics/CLIP` tokenizer dependency from GitHub. If network access is blocked, upload approved detector and OpenCLIP checkpoints into the extracted project via the Files pane and set `yoloe.model` and `matching.pretrained` to those relative paths before creating the pilot config. The YOLOE text encoder/cache and tokenizer dependency must also be available; a detector checkpoint alone does not make text inference fully offline. Download all results before runtime deletion.